<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-04-rag/lesson-4.6-graph-rag/notebooks/GCP_Capstone_4.6_Graph_RAG.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.6 Graph RAG — A Tenant-Scoped Knowledge Graph on Spanner Graph
**Netsetos GenAI Engineering — GCP Capstone** · Module 4 · new in v1.1

Some questions have no answer in any single chunk. *“Who owns the system affected by the policy that supersedes ACME-STD-011?”* is three facts in three documents. Vector search retrieves passages; it cannot perform the join. This notebook builds the join:

- **structured extraction** of a typed subgraph from each chunk, on `gemini-3.1-flash-lite`
- **entity resolution** so one company written four ways becomes one node
- a **Spanner Graph property graph** whose first primary-key column is `tenant_id`
- **GQL traversal** one or two hops out, tenant-scoped and capped
- a **`retrieval_mode` router** (vector | graph | auto) and one JSON log line per query
- a **NetworkX fallback** with the same contract, for when there is no Spanner instance
- **two more backends behind the same three calls** — Firestore (no new instance) and BigQuery (a recursive CTE) — because a graph this size does not need a graph database, and the question *can we use anything other than Spanner* deserves a measured answer, not a shrug

Prerequisites: lessons **4.1** and **4.2** have run, so Firestore `chunks` holds embedded chunks — ACME's 1,624 of them: the handbook, contracts, invoice, report and thirteen real documents. The graph stores chunk ids, not text — the corpus stays exactly where 4.1 put it.

**Cost:** the Spanner free trial instance is ₹0 for 90 days (Enterprise edition, which is what Spanner Graph requires). The only real spend is the extraction pass, on the cheapest model. *Facts verified 2026-09-03.*

## Setup
Two clients, as in every lesson since 1.1: Gemini 3.x generation is served **only** from `global`; embeddings are **regional-only**. Entity resolution embeds, so it needs the regional client.

In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-spanner==3.60.0 \
                 google-cloud-firestore==2.30.0 networkx==3.5

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"     # CHANGE THIS
TENANT     = "acme"                    # every node and edge carries this
SPANNER_INSTANCE = "documind-graph"
SPANNER_DATABASE = "documind"
SPANNER_REGION   = "regional-asia-south1"   # India data residency; graph data at rest stays in Mumbai

# The graph is three calls - load, seed, expand - and a tenant delete, and Cell 1b gives them
# three backends. "firestore" keeps the graph in the chunks' own database: nothing to provision,
# nothing that expires - the demo's choice, and the lane's. "spanner" is the GQL lesson (a free
# trial that ends; Cells 0b and 1 run only for it). "bigquery" is an edges table and a recursive
# CTE whose SQL also runs on Cloud SQL Postgres; it needs 5.5's dataset, which the lean lane does
# not create: `bq mk --location=asia-south1 rag_data` first.
GRAPH_BACKEND = "firestore"               # "spanner" | "firestore" | "bigquery"
BQ_DATASET    = "rag_data"                # the BigQuery backend: 5.5's dataset, in asia-south1

import re, subprocess
from google import genai
from google.genai import types
from google.cloud import firestore

# Two clients, same rule as 4.5: Gemini 3.x generation is served ONLY from global;
# embeddings are regional-only. Entity resolution embeds, so it needs the regional one.
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location="global")
emb_client = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")

db = firestore.Client(project=PROJECT_ID)   # the chunks (4.1-4.5) and, on this backend, the graph

EXTRACT_MODEL = "gemini-3.1-flash-lite"   # bulk extraction: cheapest per token
ANSWER_MODEL  = "gemini-3.6-flash"
EMBED_MODEL   = "text-embedding-005"      # same model as chunks (4.1-4.2) - never mix
print("clients ready")

In [ ]:
import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


### Create the graph database
A **free trial** Spanner instance: 90 days, 10 GB, at most 5 databases, one per project. It defaults to the **Enterprise edition**, which is the edition Spanner Graph requires.

It is created in `regional-asia-south1` (Mumbai): the graph holds entities extracted from customer documents, so it is customer data at rest and stays in India — the same rule lesson 2.3 applied to Firestore.

> **Write down the two dates this cell prints.** After 90 days the instance stops serving and enters a 30-day grace period; after that the instance and its data are deleted.

In [ ]:
# A Spanner FREE TRIAL instance: 90 days, 10 GB, at most 5 databases, one per project.
# It defaults to Enterprise edition, which is what Spanner Graph requires - so the whole
# lesson runs at no cost. Verified on the free-trial-instance page 2026-09-03.
# Runs only for GRAPH_BACKEND == "spanner": the lane has no Spanner and enables nothing for it.
if GRAPH_BACKEND == "spanner":
    for cmd in (["gcloud", "services", "enable", "spanner.googleapis.com", "--project", PROJECT_ID],
                ["gcloud", "spanner", "instances", "create", SPANNER_INSTANCE, "--config", SPANNER_REGION,
                 "--description", "DocuMind knowledge graph", "--instance-type", "free-instance", "--project", PROJECT_ID],
                ["gcloud", "spanner", "databases", "create", SPANNER_DATABASE, "--instance", SPANNER_INSTANCE, "--project", PROJECT_ID]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        err = r.stderr.strip()
        print(" ".join(cmd[1:4]), "->", "ok" if r.returncode == 0 else ("already exists - continuing" if "already exists" in err.lower() else err[-200:]))

    # WRITE THE EXPIRY DOWN. After 90 days the instance stops serving and enters a 30-day
    # grace period; after that it and its data are deleted. Re-run this notebook before then.
    import datetime
    print("trial ends:", (datetime.date.today() + datetime.timedelta(days=90)).isoformat())
    print("data deleted after:", (datetime.date.today() + datetime.timedelta(days=120)).isoformat())
else:
    print(f"GRAPH_BACKEND={GRAPH_BACKEND!r}: no Spanner instance is created (Cells 0b and 1 are Spanner-only)")


## Cell 1: The schema — `tenant_id` is the first primary-key column
Multi-tenancy is a schema decision, not a `WHERE` clause you remember to add. With `tenant_id` first, one tenant's subgraph is a contiguous key range: reads are local, and a query that loses its filter is a scan you notice rather than a silent cross-tenant leak.

`GraphEdge` is **interleaved in** `GraphNode` with `ON DELETE CASCADE`, so edges sit next to their source node and deleting a tenant's nodes deletes its edges in the same transaction — the erasure path.

`CREATE PROPERTY GRAPH` copies nothing. It declares a graph *view* over the two tables: which supplies nodes, which supplies edges, and how an edge joins to its endpoints. The rows stay queryable with ordinary SQL.

In [ ]:
from google.cloud import spanner

spanner_client = spanner.Client(project=PROJECT_ID)
instance = spanner_client.instance(SPANNER_INSTANCE)
database = instance.database(SPANNER_DATABASE)

# tenant_id is the FIRST primary-key column on both tables. Every query filters on it,
# every row carries it, and edges are interleaved in their source node so a tenant's
# subgraph is physically co-located and a tenant delete cascades.
# CREATE TABLE IF NOT EXISTS is valid Spanner DDL, so this cell is safe to re-run
# (verified 2026-09-03). CREATE OR REPLACE re-declares the property graph in place.
DDL = [
    """CREATE TABLE IF NOT EXISTS GraphNode (
         tenant_id  STRING(64)  NOT NULL,
         node_id    STRING(128) NOT NULL,
         kind       STRING(64)  NOT NULL,   -- entity type. NOT called `label`: LABEL is a GQL keyword
         name       STRING(MAX) NOT NULL,
         chunk_ids  ARRAY<STRING(128)>,
         updated_at TIMESTAMP OPTIONS (allow_commit_timestamp=true),
       ) PRIMARY KEY (tenant_id, node_id)""",

    """CREATE TABLE IF NOT EXISTS GraphEdge (
         tenant_id  STRING(64)  NOT NULL,
         node_id    STRING(128) NOT NULL,
         dst_id     STRING(128) NOT NULL,
         rel        STRING(64)  NOT NULL,
         chunk_id   STRING(128),
         confidence FLOAT64,
       ) PRIMARY KEY (tenant_id, node_id, dst_id, rel),
         INTERLEAVE IN PARENT GraphNode ON DELETE CASCADE""",

    """CREATE OR REPLACE PROPERTY GRAPH DocuMindGraph
         NODE TABLES (GraphNode)
         EDGE TABLES (
           GraphEdge
             SOURCE KEY (tenant_id, node_id) REFERENCES GraphNode (tenant_id, node_id)
             DESTINATION KEY (tenant_id, dst_id) REFERENCES GraphNode (tenant_id, node_id)
             LABEL RELATES_TO
         )""",
]

if GRAPH_BACKEND == "spanner":
    database.update_ddl(DDL).result(timeout=300)
    print("property graph DocuMindGraph ready")
else:
    print(f"GRAPH_BACKEND={GRAPH_BACKEND!r}: Spanner DDL skipped")

## Cell 1b: Other than Spanner — the same three calls on Firestore and BigQuery

Every backend below implements `load(nodes, edges)`, `seed(question)`, `expand(seed_ids, hops, cap)` and `delete_tenant()`. `retrieve()` in Cell 9 calls `GRAPH.seed()` and `GRAPH.expand()` and never learns which store answered — the same discipline as `retrieve()` in Modules 6–8.

| Backend | Where the graph lives | Traversal | New infrastructure | Expiry | Tenant delete | India residency |
|---|---|---|---|---|---|---|
| **Spanner Graph** | `GraphNode` / `GraphEdge`, edges interleaved | GQL `MATCH … -[e]-{1,2}` | a Spanner instance (free trial) | 90 days on the trial; a paid instance after | one key-range delete, cascade | `regional-asia-south1` |
| **Firestore** | `graph_nodes` / `graph_edges` documents, next to `chunks` | a Python walk: one `in` query per hop and direction | none — the database 2.3 created | none | a batch delete over the tenant's documents | the `(default)` database is in `asia-south1` |
| **BigQuery** | two tables in `rag_data` (5.5's dataset) | `WITH RECURSIVE` — standard SQL; the same statement runs on Cloud SQL Postgres (12.8's checkpoint instance) | none — the dataset 5.5 created | none | `DELETE … WHERE tenant_id = …` | the dataset is in `asia-south1` |
| **NetworkX** (Cell 11) | a pickle | in memory | none | none | drop the file | wherever the notebook runs |

A graph extracted from a few thousand chunks has a few thousand nodes. At that size the question is not *which engine traverses fastest* but *which store you already run, who deletes a tenant, and when the free tier ends* — which is why the column that matters most is **New infrastructure**. Spanner Graph earns its place when the graph is large, hot and queried in GQL by many services; for DocuMind's size, Firestore is the honest default and BigQuery the one that also feeds 5.5's SQL lane.


In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

_STOP = {"what", "which", "who", "whose", "how", "when", "where", "why", "is", "are", "does", "do",
         "can", "the", "if", "in", "under", "after", "before", "which acts", "what did"}

def _candidate_names(question: str) -> list:
    """Capitalised runs in the question, lower-cased - a cheap proper-noun guess, no model call.
    A run may stop short of the entity's full name ("Code on Wages" for "Code on Wages, 2019"),
    so every backend's seed() matches by CONTAINMENT: the node's name inside the question, or a
    candidate inside the node's name. An exact match on the full lower-cased name found nothing
    on the real corpus - the extracted names carry the year and the brackets."""
    runs = re.findall(r"\b[A-Z][\w&-]*(?:\s+(?:[A-Z][\w&-]*|of|on|and|for))*", question)
    return [r.lower() for r in runs if r.lower() not in _STOP and len(r) >= 5] or [question.lower()]


def _seed_match(name_lower: str, question_lower: str, candidates: list) -> bool:
    return bool(name_lower) and (name_lower in question_lower or any(c in name_lower for c in candidates))


class SpannerGraph:
    """This lesson's backend, behind the three-call interface. The methods are Cells 5-7 and the
    cleanup cell, so this class is thin on purpose."""

    def load(self, nodes: dict, edges: dict, tenant_id: str = TENANT) -> None:
        load_graph(nodes, edges, tenant_id)                      # Cell 5

    def seed(self, question: str, tenant_id: str = TENANT, limit: int = 5) -> list:
        return seed_nodes(question, tenant_id, limit)            # Cell 6

    def expand(self, seed_ids: list, tenant_id: str = TENANT, hops: int = 1, cap: int = 20) -> list:
        return graph_expand(seed_ids, tenant_id, hops, cap)      # Cell 7

    def delete_tenant(self, tenant_id: str) -> None:
        delete_tenant(tenant_id)                                 # Cleanup


class FirestoreGraph:
    """Nodes and edges as documents in graph_nodes / graph_edges, keyed tenant:node_id, in the
    database the chunks already live in. A hop is one `in` query per direction over at most 30
    ids (Firestore's limit on `in`); two hops are two rounds. Nothing to provision, nothing that
    expires, the tenant predicate in every read, and a tenant delete is a batch over the
    tenant's documents - the DPDP erasure path, as a query rather than a schema property."""

    def __init__(self, db):
        self.db = db

    def load(self, nodes: dict, edges: dict, tenant_id: str = TENANT) -> None:
        batch, n = self.db.batch(), 0
        for nid, node in nodes.items():
            batch.set(self.db.collection("graph_nodes").document(f"{tenant_id}:{nid}"),
                      {"tenant_id": tenant_id, "node_id": nid, "kind": node["kind"], "name": node["name"],
                       "name_lower": node["name"].lower(), "chunk_ids": sorted(node["chunks"])})
            n += 1
            if n % 400 == 0:
                batch.commit(); batch = self.db.batch()
        for (s, d, rel), v in edges.items():
            batch.set(self.db.collection("graph_edges").document(f"{tenant_id}:{s}:{d}:{rel}"),
                      {"tenant_id": tenant_id, "node_id": s, "dst_id": d, "rel": rel,
                       "chunk_id": v["chunk_id"], "confidence": v["confidence"]})
            n += 1
            if n % 400 == 0:
                batch.commit(); batch = self.db.batch()
        batch.commit()
        print(f"{len(nodes)} nodes, {len(edges)} edges written for tenant {tenant_id} (Firestore)")

    def seed(self, question: str, tenant_id: str = TENANT, limit: int = 5) -> list:
        """One pass over the tenant's nodes (a few thousand documents at most), matched by
        containment; the most specific name first. Production would keep a name-token index."""
        cands, ql = _candidate_names(question), question.lower()
        hits = []
        for d in (self.db.collection("graph_nodes").where(filter=FieldFilter("tenant_id", "==", tenant_id))
                  .select(["node_id", "name", "kind", "name_lower"]).stream()):
            if _seed_match(d.get("name_lower") or "", ql, cands):
                hits.append({"node_id": d.get("node_id"), "name": d.get("name"), "kind": d.get("kind")})
        hits.sort(key=lambda n: -len(n["name"] or ""))
        return hits[:limit]

    def expand(self, seed_ids: list, tenant_id: str = TENANT, hops: int = 1, cap: int = 20) -> list:
        if hops not in (1, 2):
            raise ValueError("hops must be 1 or 2 - deeper walks return the whole tenant")
        frontier, seen = set(seed_ids), set(seed_ids)
        for _ in range(hops):
            nxt = set()
            ids = sorted(frontier)
            for i in range(0, len(ids), 30):                       # `in` takes at most 30 values
                for field, other in (("node_id", "dst_id"), ("dst_id", "node_id")):   # undirected, like GQL's -[e]-
                    q = (self.db.collection("graph_edges")
                         .where(filter=FieldFilter("tenant_id", "==", tenant_id))
                         .where(filter=FieldFilter(field, "in", ids[i:i + 30])))
                    nxt |= {e.get(other) for e in q.stream()}
            frontier = nxt - seen
            seen |= nxt
        out = []
        ids = sorted(seen)
        for i in range(0, len(ids), 30):
            q = (self.db.collection("graph_nodes")
                 .where(filter=FieldFilter("tenant_id", "==", tenant_id))
                 .where(filter=FieldFilter("node_id", "in", ids[i:i + 30])))
            out += [{"node_id": d.get("node_id"), "name": d.get("name"), "kind": d.get("kind"),
                     "chunk_ids": list(d.get("chunk_ids") or [])} for d in q.stream()]
        return sorted(out, key=lambda n: n["name"])[:cap]

    def delete_tenant(self, tenant_id: str) -> None:
        for coll in ("graph_edges", "graph_nodes"):
            docs = list(self.db.collection(coll).where(filter=FieldFilter("tenant_id", "==", tenant_id)).stream())
            batch, n = self.db.batch(), 0
            for d in docs:
                batch.delete(d.reference); n += 1
                if n % 400 == 0:
                    batch.commit(); batch = self.db.batch()
            batch.commit()
            print(f"tenant {tenant_id}: {len(docs)} {coll} deleted")


class BigQueryGraph:
    """An edges table and a recursive CTE, in 5.5's dataset. The SQL is standard - WITH RECURSIVE,
    a self-join, DISTINCT - so the same statement runs on BigQuery, on Cloud SQL Postgres (12.8's
    checkpoint instance) and, which is how this class is tested offline, on sqlite. BigQuery has
    no upsert, so load() deletes the tenant's rows and loads afresh: idempotent by construction."""

    DDL = ["CREATE TABLE IF NOT EXISTS {nodes} (tenant_id STRING, node_id STRING, kind STRING, name STRING, name_lower STRING, chunk_ids STRING)",
           "CREATE TABLE IF NOT EXISTS {edges} (tenant_id STRING, node_id STRING, dst_id STRING, rel STRING, chunk_id STRING, confidence FLOAT64)"]
    # LIKE rather than STRPOS: the same statement runs on BigQuery, on Cloud SQL Postgres and on sqlite
    SEED = ("SELECT node_id, name, kind FROM {nodes} WHERE tenant_id = @tenant "
            "AND (@question LIKE '%' || name_lower || '%'{cands}) ORDER BY LENGTH(name) DESC LIMIT {limit}")
    EXPAND = """
WITH RECURSIVE walk AS (
  SELECT node_id, 0 AS depth FROM {nodes} WHERE tenant_id = @tenant AND node_id IN ({seeds})
  UNION ALL
  SELECT CASE WHEN e.node_id = w.node_id THEN e.dst_id ELSE e.node_id END AS node_id, w.depth + 1 AS depth
  FROM walk w JOIN {edges} e
    ON e.tenant_id = @tenant AND (e.node_id = w.node_id OR e.dst_id = w.node_id)
  WHERE w.depth < @hops
)
SELECT DISTINCT n.node_id, n.name, n.kind, n.chunk_ids
FROM walk w JOIN {nodes} n ON n.tenant_id = @tenant AND n.node_id = w.node_id
ORDER BY n.name
LIMIT {cap}"""

    def __init__(self, client, dataset: str = BQ_DATASET, project: str = PROJECT_ID):
        self.client = client
        self.nodes = f"`{project}.{dataset}.graph_nodes`"
        self.edges = f"`{project}.{dataset}.graph_edges`"
        for ddl in self.DDL:
            client.query(ddl.format(nodes=self.nodes, edges=self.edges)).result()

    @staticmethod
    def _params(values: dict):
        from google.cloud import bigquery
        return [bigquery.ScalarQueryParameter(k, "INT64" if isinstance(v, int) else "STRING", v)
                for k, v in values.items()]

    def _run(self, sql: str, values: dict):
        from google.cloud import bigquery
        job = self.client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=self._params(values)))
        return [dict(row) for row in job.result()]

    def load(self, nodes: dict, edges: dict, tenant_id: str = TENANT) -> None:
        self.delete_tenant(tenant_id)
        self.client.load_table_from_json(
            [{"tenant_id": tenant_id, "node_id": nid, "kind": n["kind"], "name": n["name"],
              "name_lower": n["name"].lower(), "chunk_ids": ",".join(sorted(n["chunks"]))} for nid, n in nodes.items()],
            self.nodes.strip("`")).result()
        self.client.load_table_from_json(
            [{"tenant_id": tenant_id, "node_id": s, "dst_id": d, "rel": rel,
              "chunk_id": v["chunk_id"], "confidence": v["confidence"]} for (s, d, rel), v in edges.items()],
            self.edges.strip("`")).result()
        print(f"{len(nodes)} nodes, {len(edges)} edges written for tenant {tenant_id} (BigQuery)")

    def seed(self, question: str, tenant_id: str = TENANT, limit: int = 5) -> list:
        cands = _candidate_names(question)
        values = {"tenant": tenant_id, "question": question.lower(), **{f"n{i}": v for i, v in enumerate(cands)}}
        sql = self.SEED.format(nodes=self.nodes, limit=int(limit),
                               cands="".join(f" OR name_lower LIKE '%' || @n{i} || '%'" for i in range(len(cands))))
        return self._run(sql, values)

    def expand(self, seed_ids: list, tenant_id: str = TENANT, hops: int = 1, cap: int = 20) -> list:
        if hops not in (1, 2):
            raise ValueError("hops must be 1 or 2 - deeper walks return the whole tenant")
        if not seed_ids:
            return []
        values = {"tenant": tenant_id, "hops": int(hops), **{f"s{i}": v for i, v in enumerate(seed_ids)}}
        sql = self.EXPAND.format(nodes=self.nodes, edges=self.edges,
                                 seeds=", ".join(f"@s{i}" for i in range(len(seed_ids))), cap=int(cap))
        return [{"node_id": r["node_id"], "name": r["name"], "kind": r["kind"],
                 "chunk_ids": [c for c in (r["chunk_ids"] or "").split(",") if c]} for r in self._run(sql, values)]

    def delete_tenant(self, tenant_id: str) -> None:
        for table in (self.edges, self.nodes):
            self._run(f"DELETE FROM {table} WHERE tenant_id = @tenant", {"tenant": tenant_id})
        print(f"tenant {tenant_id}: graph rows deleted (BigQuery)")


if GRAPH_BACKEND == "spanner":
    GRAPH = SpannerGraph()
elif GRAPH_BACKEND == "firestore":
    GRAPH = FirestoreGraph(db)                          # the chunks' own database (Cell 0)
elif GRAPH_BACKEND == "bigquery":
    from google.cloud import bigquery
    GRAPH = BigQueryGraph(bigquery.Client(project=PROJECT_ID))
else:
    raise ValueError(f"GRAPH_BACKEND must be spanner, firestore or bigquery, not {GRAPH_BACKEND!r}")
print(f"graph backend: {type(GRAPH).__name__}")


## Cell 2: Extraction — a typed subgraph from each passage
Structured output turns the model into a parser. Two constraints in the system prompt do the work: **only what is stated** (no world knowledge, which is how graphs fill with unsupported facts) and **every relation's endpoints must appear in entities** (so dangling edges are detectable and the loader can drop them).

This runs on `gemini-3.1-flash-lite`: it is the highest-volume call in the course and the easiest task. `thinking_level="LOW"` requests the lowest thinking level — extraction is not reasoning.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

class Entity(BaseModel):
    name: str = Field(description="Surface form exactly as written in the text")
    type: Literal["person", "org", "product", "policy", "system", "location", "date"]

class Relation(BaseModel):
    source: str = Field(description="name of the source entity, exactly as in entities")
    target: str = Field(description="name of the target entity, exactly as in entities")
    rel: str = Field(description="UPPER_SNAKE verb phrase, e.g. OWNS, REPORTS_TO, SUPERSEDES")
    confidence: float = Field(ge=0, le=1)

class GraphExtraction(BaseModel):
    entities: List[Entity]
    relations: List[Relation]

EXTRACT_SYSTEM = """You build a knowledge graph from enterprise documents.
Extract only entities and relations STATED in the passage. Never infer, never add
world knowledge. Every relation's source and target must appear in entities.
If the passage states no relation, return an empty relations list."""

def extract_graph(chunk_text: str) -> GraphExtraction:
    """One chunk in, a typed subgraph out. Flash-Lite: this is bulk work, not reasoning."""
    r = gen_client.models.generate_content(
        model=EXTRACT_MODEL,
        contents=f"Passage:\n{chunk_text}",
        config=types.GenerateContentConfig(
            system_instruction=EXTRACT_SYSTEM,
            response_mime_type="application/json",
            response_schema=GraphExtraction,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),  # lowest level: extraction is not reasoning
        ),
    )
    return draft_of(r, GraphExtraction)

### Extract over the whole corpus
Pass the chunks in the shape 4.2 retrieves them: `[{"chunk_id": ..., "content": ...}]`. Keeping the chunk id with each extraction is the entire provenance chain: chunk → entity → edge → citation.

One failed chunk must not stop an ingest, so failures are caught, reported and skipped.

In [ ]:
import time

def extract_all(chunks: list) -> list:
    """Extract over every chunk, keeping the chunk id so each fact stays traceable.

    chunks: [{"chunk_id": str, "text": str}, ...] - the same keys 4.2 retrieves.
    """
    out = []
    for i, c in enumerate(chunks, 1):
        try:
            g = extract_graph(c["text"])
        except Exception as e:                      # one bad chunk must not stop the ingest
            print(f"  [{i}/{len(chunks)}] {c['chunk_id']}: extraction failed ({e}) - skipped")
            continue
        out.append({"chunk_id": c["chunk_id"], "graph": g})
        print(f"  [{i}/{len(chunks)}] {c['chunk_id']}: "
              f"{len(g.entities)} entities, {len(g.relations)} relations")
        time.sleep(0.1)                             # stay well inside the per-minute quota
    return out

## Cell 3: Entity resolution — one company, four spellings
Across a real corpus the same organisation appears as “ACME Corp”, “ACME Corporation”, “Acme Corp.” and “ACME”. Four nodes means the two-hop walk that should connect a policy to its owner stops at a node with no edges. Without resolution the graph does not work.

Two passes: normalise (case, punctuation, corporate suffixes), then cosine similarity over embeddings at **0.92**.

> **Why so strict?** A duplicate node makes one walk miss an edge — visible, recoverable. A wrong merge fuses two real entities and produces confident answers about a thing that does not exist — invisible, unrecoverable. When the two errors are this asymmetric, tune towards the one you can see.

In [ ]:
import re, numpy as np

def normalise(name: str) -> str:
    """Cheap first pass: case, punctuation and the corporate suffixes that create duplicates."""
    n = name.lower().strip()
    n = re.sub(r"[\.,]", "", n)
    n = re.sub(r"\b(private|pvt|limited|ltd|inc|llc|corp|corporation|co)\b", "", n)
    return re.sub(r"\s+", " ", n).strip()

def embed_names(names: list) -> np.ndarray:
    """text-embedding-005 takes up to 250 texts per request."""
    vecs = []
    for i in range(0, len(names), 250):
        r = emb_client.models.embed_content(
            model=EMBED_MODEL, contents=names[i:i + 250],
            config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY",
                                            output_dimensionality=768))
        vecs.extend(e.values for e in r.embeddings)
    v = np.array(vecs, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)   # cosine becomes a dot product

RESOLVE_THRESHOLD = 0.92   # deliberately high: a wrong merge is worse than a duplicate node

def resolve_entities(names: list) -> dict:
    """Map every surface form to one canonical name.

    Two passes: exact match after normalise(), then cosine similarity over embeddings for
    the survivors. Returns {surface_form: canonical_name}.
    """
    canon_of, buckets = {}, {}
    for n in names:                                  # pass 1 - normalise
        buckets.setdefault(normalise(n), []).append(n)
    reps = sorted(buckets)                           # one representative per normalised key
    if not reps:
        return canon_of
    vecs = embed_names([buckets[k][0] for k in reps])
    merged_into = {}
    for i in range(len(reps)):
        if reps[i] in merged_into:
            continue
        for j in range(i + 1, len(reps)):
            if reps[j] in merged_into:
                continue
            if float(vecs[i] @ vecs[j]) >= RESOLVE_THRESHOLD:   # pass 2 - semantic
                merged_into[reps[j]] = reps[i]
    for key, surfaces in buckets.items():
        target = merged_into.get(key, key)
        canonical = buckets[target][0]               # longest-lived surface form wins
        for s in surfaces:
            canon_of[s] = canonical
    return canon_of

## Cell 4: Run the ingest — extraction and resolution
This is the cell that turns the corpus into facts. It loads the tenant's chunks from `chunks` — the one collection 4.1, 4.2 and 4.5 write, every document carrying `tenant_id` — extracts a subgraph from each and resolves the surface forms **once**. Cell 5 writes the result to whichever backend `GRAPH_BACKEND` chose.

The corpus is real and it is big: ACME holds 1,624 chunks — its handbook, contracts, invoice and report, and thirteen real documents (twelve Acts and Codes of Parliament and the ministry's compliance handbook). Extraction is one Flash-Lite call per chunk, so `load_corpus()` takes documents in a stated order — the Code on Wages and the two Maternity Benefit Acts (the documents Cell 10's questions need), then the handbook, the contracts, the invoice and the report, the ministry's handbook, then the Gratuity Act — matched on the source file's name, because the ingest worker writes neither a section nor a doc type; it skips boilerplate and stops at `limit`. Two hundred chunks is a few rupees and a few minutes; the whole tenant is priced in Cell 12 before you run it.


In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

# The order documents are taken in: what the demo questions and the graph need first. Matched on
# the source file's name, because that is the one field every writer of `chunks` fills the same
# way - the ingest worker leaves doc_type "unknown" and writes no section (4.8), so an order keyed
# on those took the lowest-hash Acts first and never reached the Maternity Benefit Acts.
DOC_ORDER = ("code_on_wages_2019", "maternity_benefit_amendment_act_2017", "maternity_benefit_act_1961",
             "hr_policy_2026", "msa_acme_2026", "inv_2026_0412", "annual_report_2026",
             "labour_codes_compliance_handbook", "payment_of_gratuity_act_1972")

def _doc_rank(source_uri: str) -> int:
    name = source_uri.rsplit("/", 1)[-1].rsplit(".", 1)[0].lower()
    return next((i for i, slug in enumerate(DOC_ORDER) if name.startswith(slug)), len(DOC_ORDER))

def _chunk_no(chunk_id: str) -> int:
    tail = chunk_id.rsplit("#", 1)[-1]
    return int(tail) if tail.isdigit() else 0

def load_corpus(tenant_id: str = TENANT, limit: int = 200) -> list:
    """The tenant's chunks from chunks - the ONE collection 4.1, 4.2 and 4.5 write, every document
    tagged with tenant_id - in DOC_ORDER, document by document, in reading order inside each; the
    handbook's GEN-xxx boilerplate and non-text chunks are skipped. `limit` is the extraction
    bill: 200 chunks (the first eight documents and part of the ninth) is a few rupees; ACME's
    full 1,624 is priced in Cell 12. The embedding field is not fetched - it is 768 floats a chunk."""
    rows, skipped = [], 0
    q = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant_id))
         .select(["text", "source_uri", "doc_type", "section", "kind"]))
    for d in q.stream():
        x = d.to_dict()
        if (x.get("section") or "").startswith("GEN-") or x.get("kind", "text") != "text":
            skipped += 1
            continue
        rows.append({"chunk_id": d.id, "text": x.get("text", ""), "source_uri": x.get("source_uri", ""),
                     "doc_type": x.get("doc_type", "")})
    rows.sort(key=lambda r: (_doc_rank(r["source_uri"]), r["source_uri"], _chunk_no(r["chunk_id"]), r["chunk_id"]))
    print(f"{len(rows)} chunks for tenant {tenant_id} ({skipped} skipped); extracting the first {min(limit, len(rows))}:",
          ", ".join(sorted({r["source_uri"].rsplit("/", 1)[-1] for r in rows[:limit]})))
    return rows[:limit]


# --- Run the ingest. This is the cell that turns the corpus into a graph. ---
chunks = load_corpus(TENANT)
if not chunks:
    raise RuntimeError("chunks is empty for this tenant - run lesson 4.2 (or 4.1) first so there is a corpus.")

extractions = extract_all(chunks)

# Resolve once, then reuse the mapping for both the Spanner load and the NetworkX fallback.
canon_of = resolve_entities([e.name for x in extractions for e in x["graph"].entities])
print(f"{len(canon_of)} surface forms -> {len(set(canon_of.values()))} canonical entities")


## Cell 5: Load the graph
`build_graph()` turns the extractions into nodes and edges: node ids are a hash of the **canonical** name, so re-ingesting a document updates rows instead of duplicating them; edges keep the highest-confidence statement when two chunks assert the same relationship; self-edges and dangling edges are dropped before they reach any store. `load_graph()` is the Spanner write — `GRAPH.load()` routes to it, or to Firestore or BigQuery, per `GRAPH_BACKEND`.


In [ ]:
import hashlib
from google.cloud.spanner_v1 import COMMIT_TIMESTAMP

def node_id(canonical_name: str) -> str:
    """Stable id from the canonical name, so re-ingesting a document updates rather than duplicates."""
    return hashlib.sha1(canonical_name.encode("utf-8")).hexdigest()[:32]

def build_graph(extractions: list, canon_of: dict | None = None) -> tuple:
    """Nodes and edges from the extractions - pure Python, no store. Pass canon_of to reuse the
    mapping you already built; omit it and one is computed here (a second embedding pass)."""
    if canon_of is None:
        canon_of = resolve_entities([e.name for x in extractions for e in x["graph"].entities])

    nodes, edges = {}, {}
    for x in extractions:
        cid, g = x["chunk_id"], x["graph"]
        for e in g.entities:
            canonical = canon_of.get(e.name, e.name)
            nid = node_id(canonical)
            n = nodes.setdefault(nid, {"name": canonical, "kind": e.type, "chunks": set()})
            n["chunks"].add(cid)                     # this is what makes citations possible
        for r in g.relations:
            src, dst = canon_of.get(r.source), canon_of.get(r.target)
            if not src or not dst or src == dst:     # drop dangling and self edges
                continue
            key = (node_id(src), node_id(dst), r.rel)
            prev = edges.get(key)
            if prev is None or r.confidence > prev["confidence"]:
                edges[key] = {"chunk_id": cid, "confidence": float(r.confidence)}
    return nodes, edges


def load_graph(nodes: dict, edges: dict, tenant_id: str = TENANT) -> None:
    """The Spanner write. Idempotent: insert_or_update on stable ids."""
    with database.batch() as batch:
        batch.insert_or_update(
            table="GraphNode",
            columns=("tenant_id", "node_id", "kind", "name", "chunk_ids", "updated_at"),
            values=[(tenant_id, nid, n["kind"], n["name"], sorted(n["chunks"]), COMMIT_TIMESTAMP)
                    for nid, n in nodes.items()])
        batch.insert_or_update(
            table="GraphEdge",
            columns=("tenant_id", "node_id", "dst_id", "rel", "chunk_id", "confidence"),
            values=[(tenant_id, s, d, rel, v["chunk_id"], v["confidence"])
                    for (s, d, rel), v in edges.items()])
    print(f"{len(nodes)} nodes, {len(edges)} edges written for tenant {tenant_id} (Spanner)")


nodes, edges = build_graph(extractions, canon_of)
GRAPH.load(nodes, edges, TENANT)                    # Spanner, Firestore or BigQuery - per GRAPH_BACKEND


## Cell 6: Find the seed entities
The graph is for **relationships**. Finding the entity a question starts from is a lookup, and 4.5's hybrid retriever already does the semantic half — so this stays deliberately simple and costs nothing.

In [ ]:
def seed_nodes(question: str, tenant_id: str = TENANT, limit: int = 5) -> list:
    """Find the entities a question is about: a containment match inside the tenant - the
    node's name inside the question, or a capitalised run of the question inside the name.

    This is deliberately simple. The graph is for RELATIONSHIPS; finding the starting
    entity is a lookup, and 4.5's hybrid retriever already does the semantic half.
    """
    cands = _candidate_names(question)
    with database.snapshot() as snapshot:
        rows = snapshot.execute_sql(
            """SELECT node_id, name, kind FROM GraphNode
               WHERE tenant_id = @tenant
                 AND (STRPOS(@question, LOWER(name)) > 0
                      OR EXISTS (SELECT 1 FROM UNNEST(@names) AS n WHERE STRPOS(LOWER(name), n) > 0))
               ORDER BY LENGTH(name) DESC
               LIMIT @limit""",
            params={"tenant": tenant_id, "question": question.lower(), "names": cands, "limit": limit},
            param_types={"tenant": param_types.STRING, "question": param_types.STRING,
                         "names": param_types.Array(param_types.STRING), "limit": param_types.INT64},
        )
        return [{"node_id": r[0], "name": r[1], "kind": r[2]} for r in rows]


## Cell 7: Traversal — one or two hops, inside one tenant, capped
The quantifier `{1,2}` after the edge pattern is the whole feature: it matches paths one or two edges long in a single query, which is the join no vector index can perform. Both node patterns filter on `tenant_id`, so the walk cannot leave the tenant even mid-path.

> **Why `hops` is restricted and results are capped.** Graph neighbourhoods grow geometrically. Three hops from a well-connected entity reaches most of the tenant, replacing a focused retrieval with a dump — and lesson 4.5 spent a whole lesson proving the context window is a budget.

In [ ]:
from google.cloud.spanner_v1 import param_types

GRAPH_EXPAND = """
GRAPH DocuMindGraph
MATCH (a:GraphNode WHERE a.tenant_id = @tenant AND a.node_id IN UNNEST(@seeds))
      -[e:RELATES_TO]-{1,%d}
      (b:GraphNode WHERE b.tenant_id = @tenant)
RETURN DISTINCT b.node_id AS node_id, b.name AS name,
                b.kind AS kind, b.chunk_ids AS chunk_ids
LIMIT @cap
"""

def graph_expand(seed_ids: list, tenant_id: str = TENANT, hops: int = 1, cap: int = 20) -> list:
    """Walk 1-2 hops out from the seed entities, inside one tenant, capped.

    The cap is not decoration: an unbounded traversal on a well-connected graph returns the
    whole tenant, which blows the context budget 4.5 spent the lesson defending.
    """
    if hops not in (1, 2):
        raise ValueError("hops must be 1 or 2 - deeper walks return the whole tenant")
    if not seed_ids:
        return []
    sql = GRAPH_EXPAND % hops
    with database.snapshot() as snapshot:
        rows = snapshot.execute_sql(
            sql,
            params={"tenant": tenant_id, "seeds": list(seed_ids), "cap": cap},
            param_types={"tenant": param_types.STRING,
                         "seeds": param_types.Array(param_types.STRING),
                         "cap": param_types.INT64},
        )
        return [{"node_id": r[0], "name": r[1], "kind": r[2], "chunk_ids": list(r[3] or [])}
                for r in rows]

## Cell 8: Fetch the chunks the graph pointed at
The graph stores chunk ids, not text. One copy of the corpus, in Firestore, exactly as 4.1 and 4.2 left it — which is what makes a graph answer citable to a document a human can open.

In [ ]:
from google.cloud import firestore
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure

db = firestore.Client(project=PROJECT_ID)

def fetch_chunks(chunk_ids: list, tenant_id: str = TENANT, limit: int = 20) -> list:
    """Pull the chunks the graph pointed at, straight from chunks by document id.

    The graph stores chunk ids, not text: one copy of the corpus, in Firestore, exactly as
    lessons 4.1 and 4.2 left it.
    """
    out = []
    for cid in chunk_ids[:limit]:
        snap = db.collection("chunks").document(cid).get()
        if snap.exists:
            d = snap.to_dict()
            out.append({"chunk_id": cid, "text": d.get("text", ""),
                        "source_uri": d.get("source_uri", "unknown"),
                        "page_start": d.get("page_start"),
                        "score": 1.0})        # fetched by id: an exact hit, not a nearest neighbour
    return out


def embed_query(q: str) -> list:
    return emb_client.models.embed_content(
        model=EMBED_MODEL, contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768),
    ).embeddings[0].values


def vector_search(question: str, tenant_id: str = TENANT, k: int = 8) -> list:
    """The vector path: the tenant-filtered find_nearest that 4.5's dense_search runs, returning
    the canonical chunk keys. With 4.5's cells loaded, hybrid_retrieve() + rerank() drop in for
    it; this one keeps the notebook runnable on its own."""
    docs = (db.collection("chunks")
              .where(filter=FieldFilter("tenant_id", "==", tenant_id))
              .find_nearest(vector_field="embedding", query_vector=Vector(embed_query(question)),
                            distance_measure=DistanceMeasure.COSINE, limit=k,
                            distance_result_field="vector_distance")
              .get())
    out = []
    for d in docs:
        x = d.to_dict()
        out.append({"chunk_id": d.id, "text": x.get("text", ""),
                    "source_uri": x.get("source_uri", "unknown"),
                    "page_start": x.get("page_start"),
                    "score": round(1 - x.get("vector_distance", 1.0), 4)})
    return out

## Cell 9: Routing — `retrieval_mode` vector, graph or auto
`auto` uses the graph only when the question is phrased relationally **and** a seed entity was found in this tenant's graph. Either alone is not enough.

Note what the router does *not* do: call a model. Lesson 3.3 built a classifier because its tiers differ in cost by 8x; here the two paths cost about the same, so a classifier would cost more than the decision is worth.

When the graph expands to nothing usable, `retrieve()` falls back to vector results and labels the mode `"vector (graph empty)"` — the user gets an answer, and the label lands in the log so an operator can see how often the graph fails to contribute.

> `vector_search()` is the tenant-filtered `find_nearest` from 4.5's dense path, defined in the cell above so this notebook runs on its own. With 4.5's cells loaded, `hybrid_retrieve()` + `rerank()` drop in for it.

In [ ]:
RETRIEVAL_MODES = ("vector", "graph", "auto")

def choose_mode(question: str, seeds: list) -> str:
    """auto: use the graph only when the question is relational AND we found a seed entity.

    No model call - a classifier here would cost more than the retrieval it is choosing.
    """
    relational = re.search(
        r"\b(who|which|whose|related|relationship|connect|between|depend|owns?|reports? to|"
        r"supersed|replac|affect|impact|downstream|upstream)\b", question, re.I)
    return "graph" if (relational and seeds) else "vector"

def retrieve(question: str, tenant_id: str = TENANT,
             retrieval_mode: str = "auto", hops: int = 1, cap: int = 20) -> dict:
    """Return chunks plus the mode that produced them, so the log line can explain the answer."""
    if retrieval_mode not in RETRIEVAL_MODES:
        raise ValueError(f"retrieval_mode must be one of {RETRIEVAL_MODES}")
    seeds = GRAPH.seed(question, tenant_id) if retrieval_mode in ("graph", "auto") else []
    mode = choose_mode(question, seeds) if retrieval_mode == "auto" else retrieval_mode

    if mode == "vector":
        return {"mode": "vector", "seeds": [], "nodes": [],
                "chunks": vector_search(question, tenant_id)}     # the tenant-filtered vector path

    nodes = GRAPH.expand([s["node_id"] for s in seeds], tenant_id, hops=hops, cap=cap)
    chunk_ids = {cid for n in nodes for cid in n["chunk_ids"]}
    chunks = fetch_chunks(sorted(chunk_ids), tenant_id)
    if not chunks:                                   # graph found nothing usable - do not fail
        return {"mode": "vector (graph empty)", "seeds": seeds, "nodes": nodes,
                "chunks": vector_search(question, tenant_id)}
    return {"mode": "graph", "seeds": seeds, "nodes": nodes, "chunks": chunks}

## Cell 10: Answer with cross-document citations, and log the mode
Every claim is cited, and each `[Source N]` is mapped back to a real Firestore chunk id. The JSON log line is what lesson 4.7 scores — grouped by `retrieval_mode`, which is how you find out whether the graph earned its place — and what lesson 12.6 aggregates per tenant.

In [ ]:
import json, time

# THE answer contract - copied verbatim from deploy/shared/documind_schemas.py at build time,
# the same text 3.2, 4.2, 4.5 and the rag-api service carry. A graph answer is a RAGAnswer:
# same Citations, same confidence, same answerable. What the graph adds goes on the DRAFT.
class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

class GraphDraft(ModelDraft):
    """The draft the model fills, plus the one thing a graph answer adds: how many hops it took."""
    hops_used: int = Field(description="0 for a vector answer, 1 or 2 for a graph answer")

ANSWER_SYSTEM = """You are DocuMind. Answer ONLY from the numbered context.
Cite every claim with [Source N]. When the answer joins facts from more than one source,
cite each one. If the context does not contain the answer, say so, set answerable=false
and confidence low. A quote is the clause that answers - at most twenty-five words, never a whole section."""

def answer_with_graph(question: str, tenant_id: str = TENANT,
                      retrieval_mode: str = "auto", hops: int = 1) -> dict:
    """The whole pipeline, with one JSON log line 4.7 will score and 12.6 will aggregate."""
    t0 = time.time()
    r = retrieve(question, tenant_id, retrieval_mode=retrieval_mode, hops=hops)
    context = "\n\n".join(
        f"[Source {i}] {c['source_uri']} p.{c['page_start']}\n{c['text']}"
        for i, c in enumerate(r["chunks"], 1))

    resp = gen_client.models.generate_content(
        model=ANSWER_MODEL,
        contents=f"Context:\n{context}\n\nQuestion: {question}",
        config=types.GenerateContentConfig(
            system_instruction=ANSWER_SYSTEM,
            response_mime_type="application/json",
            response_schema=GraphDraft,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    draft: GraphDraft = draft_of(resp, GraphDraft)
    answer = resolve(draft, r["chunks"])            # [Source N] -> chunk id, source, page, score

    meta = resp.usage_metadata
    log = {"event": "graph_query", "tenant": tenant_id, "retrieval_mode": r["mode"],
           "hops": hops if r["mode"] == "graph" else 0,
           "seeds": [s["name"] for s in r["seeds"]], "nodes_expanded": len(r["nodes"]),
           "chunks": len(r["chunks"]), "citations": len(answer.citations),
           "confidence": answer.confidence, "answerable": answer.answerable,
           "prompt_tokens": meta.prompt_token_count,
           "output_tokens": meta.candidates_token_count,
           "thinking_tokens": meta.thoughts_token_count or 0,
           "latency_ms": int((time.time() - t0) * 1000)}
    print(json.dumps(log))
    return {"answer": answer, "hops_used": draft.hops_used, "log": log, "retrieval": r}

### Three questions, three shapes
A lookup (`auto` should route to vector), a one-hop relational question — the Code on Wages names the four Acts it repeals in its last section — and the multi-hop join that motivated the lesson: what the 2017 amending Act changed in section 5 of the 1961 Act lives in two documents. Watch `retrieval_mode` in the log line for each.


In [ ]:
QUESTIONS = [
    # single-hop: the graph is not needed, auto should route to vector
    "What is the overtime rate under the Code on Wages?",
    # relational: one hop from a named entity (the Code's last section names the four Acts it repeals)
    "Which Acts does the Code on Wages, 2019 repeal?",
    # multi-hop: the join no single chunk contains - the amending Act and the principal Act
    "What did the Maternity Benefit (Amendment) Act, 2017 change in section 5 of the 1961 Act?",
]

for q in QUESTIONS:
    print("\n" + "=" * 78)
    print("Q:", q)
    out = answer_with_graph(q, retrieval_mode="auto", hops=2)
    a = out["answer"]
    print(f"\nmode={out['log']['retrieval_mode']}  hops_used={out['hops_used']}  "
          f"confidence={a.confidence}  answerable={a.answerable}")
    print(a.answer[:400])
    for c in a.citations:
        print(f"  {c.chunk_id} ({c.source_uri.rsplit('/', 1)[-1]}, p.{c.page}): {c.quote[:80]}")

## Cell 11: The ladder down — NetworkX with the same contract
A learner without billing, or an instance past its 90 days, still needs to finish. `nx_expand()` returns the same dictionaries as `graph_expand()`, so `retrieve()` works against either backend unchanged.

Between Spanner and this rung sit Cell 1b's two stores you already run — Firestore and BigQuery — and, outside the project, **Neo4j AuraDB Free**: a managed property graph with no time limit, at the cost of a second vendor, data outside your project and region (so the India residency story above no longer holds) and Cypher instead of GQL.

In [ ]:
import networkx as nx, pickle

def build_networkx(extractions: list, canon_of: dict) -> nx.MultiDiGraph:
    """Rung 3: no Spanner, no cost, no cloud. Same shape, in memory."""
    G = nx.MultiDiGraph()
    for x in extractions:
        g, cid = x["graph"], x["chunk_id"]
        for e in g.entities:
            name = canon_of.get(e.name, e.name)
            if not G.has_node(name):
                G.add_node(name, kind=e.type, chunk_ids=set())
            G.nodes[name]["chunk_ids"].add(cid)
        for r in g.relations:
            s, d = canon_of.get(r.source), canon_of.get(r.target)
            if s and d and s != d:
                G.add_edge(s, d, rel=r.rel, chunk_id=cid, confidence=r.confidence)
    return G

def nx_expand(G: nx.MultiDiGraph, seed_names: list, hops: int = 1, cap: int = 20) -> list:
    """The graph_expand() contract, backed by NetworkX. Undirected walk, same cap."""
    seen = set()
    for s in seed_names:
        if s in G:
            seen |= nx.single_source_shortest_path_length(G.to_undirected(as_view=True),
                                                          s, cutoff=hops).keys()
    return [{"node_id": n, "name": n, "kind": G.nodes[n].get("kind", ""),
             "chunk_ids": sorted(G.nodes[n].get("chunk_ids", []))}
            for n in sorted(seen)][:cap]

# Persist it so the graph survives a runtime restart without re-paying for extraction.
with open("documind_graph.pkl", "wb") as f:
    pickle.dump(build_networkx(extractions, canon_of), f)
print("NetworkX graph saved to documind_graph.pkl")

## Cell 12: Cost
Extraction is the whole bill: one Flash-Lite call per chunk, once. Spanner is ₹0 while the trial lasts.

In [ ]:
USD_INR = 85

# Per 1M tokens. Flash-Lite does the bulk extraction; 3.6 Flash answers.
PRICING = {
    "gemini-3.1-flash-lite": {"in": 0.25, "out": 1.50},
    "gemini-3.6-flash":      {"in": 1.50, "out": 7.50},   # standard rate from 2027-01-01
}
PRICING_INTRO = {                                          # through 2026-12-31
    "gemini-3.1-flash-lite": {"in": 0.25, "out": 1.50},
    "gemini-3.6-flash":      {"in": 0.75, "out": 3.75},
}

def ingest_cost(n_chunks: int, avg_in: int = 600, avg_out: int = 200, rate: str = "intro") -> float:
    """One-off cost of turning a corpus into a graph. Extraction is the whole bill.

    Flash-Lite has no introductory discount, so both rate tables give the same figure here;
    the parameter exists so the function still reads correctly if you switch the model.
    """
    p = (PRICING_INTRO if rate == "intro" else PRICING)["gemini-3.1-flash-lite"]
    usd = n_chunks * (avg_in / 1e6 * p["in"] + avg_out / 1e6 * p["out"])
    print(f"  {n_chunks:,} chunks, {rate} rate: ${usd:.2f} (Rs {usd * USD_INR:,.0f}) one-off")
    return usd

for n in (200, 1_624, 10_000):          # Cell 4's default, ACME's whole corpus, a real tenant
    ingest_cost(n)

# Spanner: the free trial instance is Rs 0 for 90 days. After that a production graph is a
# paid instance - price it on the Spanner pricing page for your region before you commit.
# Firestore and BigQuery (Cell 1b) add no instance: a few thousand document writes, or a few
# MB of table storage and a query that scans it, are rupees a month either way.

## Cleanup — and the tenant-erasure path
On Spanner, because edges are interleaved with `ON DELETE CASCADE`, deleting a tenant's nodes deletes its edges in the same transaction — the DPDP erasure path as a property of the schema. On Firestore and BigQuery it is a query over `tenant_id`, which `GRAPH.delete_tenant()` runs for you; the point is that every backend has one, and 13.2 tests it.

In [ ]:
# Tenant delete: edges are INTERLEAVED IN PARENT GraphNode ON DELETE CASCADE, so removing a
# tenant's nodes removes its edges in the same transaction. This is the DPDP erasure path.
from google.cloud.spanner_v1 import KeySet

def delete_tenant(tenant_id: str) -> None:
    with database.batch() as batch:
        batch.delete("GraphNode", KeySet(ranges=[
            spanner.KeyRange(start_closed=[tenant_id], end_closed=[tenant_id])]))
    print(f"tenant {tenant_id}: nodes and interleaved edges deleted")

GRAPH.delete_tenant(TENANT)     # whichever backend holds the graph

# Drop the whole trial instance when you are done (it would otherwise expire on its own):
# !gcloud spanner instances delete $SPANNER_INSTANCE --project=$PROJECT_ID --quiet

## ✅ Lesson 4.6 complete

- ✅ Named the question shape vector search cannot answer, and why
- ✅ Extracted typed subgraphs with structured output on the cheapest model
- ✅ Resolved surface forms into identities, and reasoned about the threshold asymmetry
- ✅ Designed a property graph with `tenant_id` as the first primary-key column
- ✅ Walked one and two hops in GQL, tenant-scoped and capped
- ✅ Routed each question with `retrieval_mode` and logged which path answered it
- ✅ Put the same three calls on Firestore and BigQuery, and said when Spanner earns its place
- ✅ Kept a NetworkX rung so the pipeline runs with no cloud graph at all

**Next: Lesson 4.7 — Evaluate Retrieval Before You Ship.** You now have three retrieval paths over one corpus and no evidence about which is better. 4.7 builds the golden set, the faithfulness gate and the comparison table that settles it, grouped by the `retrieval_mode` field you started logging today.